# E2B 单样本转换诊断
只跑原始 E2B 的 1→2→4 条算术题，不跑全量数据。当前阶段全部题在 4 个固定种子下连续两次评测答对，才进入下一阶段；每阶段最多 400 步，失败就停止。通过仅证明小样本拟合，不代表完成转换或学会一般算术。
保持 C=256、K=18、全画布平均 CE、BF16 AdamW、学习率 1.125e-4、前 50 步预热。两条未见题只评测，不决定阶段晋级。
先选 A100 80GB。上传完整包 `llamafactory_single_sample_v1.tgz`。检查点保留最近两份，含优化器状态；请在 Drive 预留至少约 100GB，保存过程中还会短暂占用本地临时空间。不要同时运行其他训练。

In [ ]:
import torch
assert torch.cuda.is_available(), '先连接 A100 80GB GPU'
free, total = torch.cuda.mem_get_info()
print(torch.cuda.get_device_name(), '总显存 GiB:', total / 1024**3, '空闲 GiB:', free / 1024**3)
assert total >= 75 * 1024**3 and free >= 45 * 1024**3, '显存不够，停止；尚未加载模型或启动训练'


In [ ]:
from google.colab import files, userdata, drive
from pathlib import Path
import os, sys, tarfile, subprocess
files.upload()  # 选择完整包 llamafactory_single_sample_v1.tgz
with tarfile.open('/content/llamafactory_single_sample_v1.tgz') as archive:
    archive.extractall('/content', filter='data')
os.environ['HF_TOKEN'] = userdata.get('HF')
os.environ['PYTHONPATH'] = '/content/src'
os.environ['DISABLE_VERSION_CHECK'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.23.0', 'safetensors', 'sentencepiece', 'pytest', 'psutil'], check=True)
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/saves/e2b_single_sample_v1'
sys.path.insert(0, '/content/scripts')
from run_single_sample_controls import preflight, fresh_destination
fresh_destination(Path(OUT))
print('资源预检:', preflight(out=OUT))
print('报告目录:', OUT)

In [ ]:
test_files = sorted(str(p) for p in Path('/content/tests/train').glob('test_diffusion_*.py'))
test_files.append('/content/tests/train/test_single_sample_controls.py')
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', *test_files, '--confcutdir=/content/tests/train'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
r.check_returncode()


In [ ]:
# 固定预算：一条题失败就停止；不要重复点击这一格。
cmd = [sys.executable, '-u', '/content/scripts/run_single_sample_controls.py', '--out', OUT]
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f'诊断退出码 {returncode}；实际报错已显示在上方。不要重复开训。')


In [ ]:
r = subprocess.run([sys.executable, '/content/scripts/run_single_sample_controls.py', '--out', OUT, '--summary-only'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)
r.check_returncode()


## 中断后的恢复
恢复时必须指定一个新的空 `--out` 目录（例如 `e2b_single_sample_v1_resumed`），再用 `--resume <旧检查点目录>`。原实验目录不会被覆盖或回滚；setup 格中的 OUT 也改成新目录。只有带 `COMPLETE` 的检查点可恢复，预算、种子和代码必须一致。脚本支持单文件或 safetensors 分片权重，并严格核对缺失/额外键；每个恢复点包含优化器、阶段步数和 RNG。
资源检查在加载模型之前执行：至少 75 GiB 总显存／45 GiB 空闲显存、32 GiB 空闲系统内存、40 GiB 本地空闲盘、100 GiB 输出盘空间。这些是保守诊断阈值，不是实测峰值；Drive FUSE 返回的空间不能替代网页里的云端配额检查。
完成后把 summary 和最后一次 EVAL/ANSWER 发回。失败阶段停止是正常的诊断结论，不代表转换通过。